# Movies Project Starter

Use this starter notebook for the BAN 6003 final project option: **The Movies Dataset / MovieLens ratings**.

Run cells from top to bottom first. Then replace starter notes with your own project decisions, checks, and interpretation.

Start each milestone by reading the matching Canvas milestone assignment page. The Canvas page tells you exactly what evidence and submission items are required; this notebook gives you starter spaces to build that work.



## Before You Start: Key Project Hints

This dataset mixes movie-level tables and rating-event-level data. The key skill is controlling granularity.

- A common final ABT is **one row per movie**.
- Do **not** directly merge raw `ratings` into `movies` and assume the result is still one row per movie.
- Aggregate ratings to `movieId` first, then merge the rating summary into the movie table.
- Use `movieId` for ratings and `tmdbId` for curated credits/keywords.
- Average rating is unstable for movies with very few ratings. Use a minimum rating-count threshold before modeling.
- Be careful about leakage: post-release fields such as revenue, popularity, vote count, or rating count may not be valid if your question is pre-release prediction.

## Setup

In [1]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists() and (REPO_ROOT.parent / "README.md").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_DIR = REPO_ROOT / "data"
OUTPUT_DIR = REPO_ROOT / "outputs"
REPORTS_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 100)


## Load Data

Read each provided table. Keep the original row meaning in mind before joining tables.

In [2]:
movies = pd.read_csv(DATA_DIR / 'movies_metadata.csv')
ratings = pd.read_csv(DATA_DIR / 'ratings.csv', parse_dates=['rating_datetime'])
links = pd.read_csv(DATA_DIR / 'links.csv')
credits = pd.read_csv(DATA_DIR / 'credits_curated.csv')
keywords = pd.read_csv(DATA_DIR / 'keywords_curated.csv')

tables = {
    'movies': movies,
    'ratings': ratings,
    'links': links,
    'credits': credits,
    'keywords': keywords,
}

## Basic Data Profile

Use this section for your first pass through row counts, columns, data types, missingness, duplicates, and suspicious values.

In [3]:
for name, df in tables.items():
    print(f'\n{name}: {df.shape[0]:,} rows x {df.shape[1]:,} columns')
    display(df.head(3))


movies: 9,082 rows x 21 columns


,movieId,tmdbId,imdb_id,title,original_title,original_language,release_date,runtime,budget,revenue,popularity,vote_average,vote_count,genres,production_companies,production_countries,adult,status,genres_list,primary_genre,release_year
0,1,862,tt0114709,Toy Story,Toy Story,en,1995-10-30,81.0,30000000,373554033.0,21.946943,7.7,5415.0,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...","[{'name': 'Pixar Animation Studios', 'id': 3}]","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Animation|Comedy|Family,Animation,1995
1,2,8844,tt0113497,Jumanji,Jumanji,en,1995-12-15,104.0,65000000,262797249.0,17.015539,6.9,2413.0,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...","[{'name': 'TriStar Pictures', 'id': 559}, {'na...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Adventure|Fantasy|Family,Adventure,1995
2,3,15602,tt0113228,Grumpier Old Men,Grumpier Old Men,en,1995-12-22,101.0,0,0.0,11.712900,6.5,92.0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...","[{'name': 'Warner Bros.', 'id': 6194}, {'name'...","[{'iso_3166_1': 'US', 'name': 'United States o...",False,Released,Romance|Comedy,Romance,1995



ratings: 99,810 rows x 5 columns


,userId,movieId,rating,timestamp,rating_datetime
0,1,31,2.5,1260759144,2009-12-14 02:52:24
1,1,1029,3.0,1260759179,2009-12-14 02:52:59
2,1,1061,3.0,1260759182,2009-12-14 02:53:02



links: 9,125 rows x 3 columns


,movieId,imdbId,tmdbId
0,1,114709,862.0
1,2,113497,8844.0
2,3,113228,15602.0



credits: 9,082 rows x 3 columns


,tmdbId,director,top_cast
0,862,John Lasseter,Tom Hanks|Tim Allen|Don Rickles
1,8844,Joe Johnston,Robin Williams|Jonathan Hyde|Kirsten Dunst
2,15602,Howard Deutch,Walter Matthau|Jack Lemmon|Ann-Margret



keywords: 9,082 rows x 2 columns


,tmdbId,keywords_list
0,862,jealousy|toy|boy|friendship|friends|rivalry|bo...
1,8844,board game|disappearance|based on children's b...
2,15602,fishing|best friend|duringcreditsstinger|old men


In [4]:
profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        'table': name,
        'rows': len(df),
        'columns': df.shape[1],
        'duplicate_rows': int(df.duplicated().sum()),
        'missing_cells': int(df.isna().sum().sum()),
    })
pd.DataFrame(profile_rows)

,table,rows,columns,duplicate_rows,missing_cells
0,movies,9082,21,0,72
1,ratings,99810,5,0,0
2,links,9125,3,0,13
3,credits,9082,3,0,114
4,keywords,9082,2,0,761


### Key and Granularity Audit

Use the expected key for each table before planning joins. `ratings` is the exception: its row meaning is a user-movie rating event, so check the combined `userId`-`movieId` key rather than expecting `movieId` alone to be unique. `links` is a crosswalk; because `movies` already contains both IDs, use it for validation unless your analysis needs one of its fields.


In [5]:
key_audit = pd.DataFrame([
    {"table": "movies", "expected_key": "movieId", "duplicate_keys": movies["movieId"].duplicated().sum(), "missing_key_values": movies["movieId"].isna().sum()},
    {"table": "movies", "expected_key": "tmdbId", "duplicate_keys": movies["tmdbId"].duplicated().sum(), "missing_key_values": movies["tmdbId"].isna().sum()},
    {"table": "ratings", "expected_key": "userId + movieId", "duplicate_keys": ratings[["userId", "movieId"]].duplicated().sum(), "missing_key_values": ratings[["userId", "movieId"]].isna().sum().sum()},
    {"table": "links", "expected_key": "movieId", "duplicate_keys": links["movieId"].duplicated().sum(), "missing_key_values": links["movieId"].isna().sum()},
    {"table": "credits", "expected_key": "tmdbId", "duplicate_keys": credits["tmdbId"].duplicated().sum(), "missing_key_values": credits["tmdbId"].isna().sum()},
    {"table": "keywords", "expected_key": "tmdbId", "duplicate_keys": keywords["tmdbId"].duplicated().sum(), "missing_key_values": keywords["tmdbId"].isna().sum()},
])

key_audit


,table,expected_key,duplicate_keys,missing_key_values
0,movies,movieId,0,0
1,movies,tmdbId,0,0
2,ratings,userId + movieId,0,0
3,links,movieId,0,0
4,credits,tmdbId,0,0
5,keywords,tmdbId,0,0


## Milestone 1 Starter: Initial Profile and Cleaning Plan

Complete this by the first project milestone.

Write notes on:

- Business problem and decision context
- Raw tables and row meaning
- Initial row and column counts
- Data type issues
- Missing values and duplicates
- Suspicious values
- Proposed target/outcome
- Early ethics or governance concerns

Important Movies notes:

- `movies` is movie-level, while `ratings` is user-movie rating-event-level.
- `credits` and `keywords` are curated movie-level tables connected by `tmdbId`.
- Budget and revenue may contain zero or missing-like values; describe this before using them.
- Decide whether your project is about audience ratings, catalog strategy, revenue, popularity, or genre/content patterns.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 1 written workspace

Use this cell to draft the narrative required by the Canvas Milestone 1 page.

- Business or research question:
- Why this question matters:
- Tables/files used:
- Row meaning of each important table:
- Early data quality issues:
- Initial cleaning plan:
- Possible target/outcome:
- Ethics, privacy, or governance concerns:



#### Milestone 1 Narrative

Business or research question: 
   - Is there a difference in perdicting which movie primary genre tend to have the highest success rate based on ratings or profits?

Why this question matters:
   - Understanding which genres tend to have higher average ratings can help movie  industry professionals make informed decisions about content creation, marketing, and viewing recommendations.

Table and Files used:
   - I am using movies and ratings tables.

Row Meaning of each important table:
   - In the movies table each row represents a movie.
   - In the ratings table each row represents a rating that a user gave to a certain movie.

Early data quality issues
   - Some columns may have missing values or duplicates. Also some movies might not have as many ratings which can affect the averages.

Inital cleaning plan:
   - I will check for missing values and duplicate. 
   - While also checking for unusal ratings and movie names.
   - While also joining the two tables using the movieId 

In [6]:
#columns with missing values
missing_movie_values=movies.isna().sum()
missing_rating_values=ratings.isna().sum()

print(f"Number of missing Movievalues: {missing_movie_values}")
print(f"Number of missing Ratingvalues: {missing_rating_values}")


Number of missing Movievalues: movieId                  0
tmdbId                   0
imdb_id                  0
title                    0
original_title           0
original_language        0
release_date             0
runtime                  0
budget                   0
revenue                  0
popularity               0
vote_average             0
vote_count               0
genres                   0
production_companies     0
production_countries     0
adult                    0
status                   2
genres_list             35
primary_genre           35
release_year             0
dtype: int64
Number of missing Ratingvalues: userId             0
movieId            0
rating             0
timestamp          0
rating_datetime    0
dtype: int64


In [7]:
#duplicated rows

full_duplicate_rows_movies = movies.duplicated().sum()
print(f"Number of fully duplicated rows in Movies: {full_duplicate_rows_movies}")

full_duplicate_rows_ratings = ratings.duplicated().sum()
print(f"Number of fully duplicated rows in Ratings: {full_duplicate_rows_ratings}")

Number of fully duplicated rows in Movies: 0
Number of fully duplicated rows in Ratings: 0


In [8]:
#minimum and maximum ratings

Minimum_Rating= ratings.min()
Maximum_Rating= ratings.max()

print("Minimum Rating", Minimum_Rating)
print("Maximum Rating", Maximum_Rating)

Minimum Rating userId                               1
movieId                              1
rating                             0.5
timestamp                    789652009
rating_datetime    1995-01-09 11:46:49
dtype: object
Maximum Rating userId                             671
movieId                         163949
rating                             5.0
timestamp                   1476640644
rating_datetime    2016-10-16 17:57:24
dtype: object


In [9]:
#movie names
movies_genres = movies["primary_genre"]
print(movies_genres)


0         Animation
1         Adventure
2           Romance
3            Comedy
4            Comedy
           ...     
9077          Drama
9078       Thriller
9079      Adventure
9080         Action
9081    Documentary
Name: primary_genre, Length: 9082, dtype: object


Possible target/outcome:
   - My main goal will be the average rating for each movie genre and see which genre has the highest rating.

Ethics, privacy, or governance concerns:
   - The ratings show viewers opinions of a few not all the viewers. User Id should only be used to analyze rating activity and should not be used to identify each user.

In [10]:
# Milestone 1 starter checks
for name, df in tables.items():
    print(f'\n{name}')
    display(df.dtypes.to_frame('dtype').head(20))
    display(df.isna().mean().sort_values(ascending=False).head(10).to_frame('missing_rate'))


movies


,dtype
movieId,int64
tmdbId,int64
imdb_id,object
title,object
original_title,object
original_language,object
release_date,object
runtime,float64
budget,int64
revenue,float64


,missing_rate
genres_list,0.003854
primary_genre,0.003854
status,0.000220
tmdbId,0.000000
movieId,0.000000
original_title,0.000000
title,0.000000
imdb_id,0.000000
original_language,0.000000
revenue,0.000000



ratings


,dtype
userId,int64
movieId,int64
rating,float64
timestamp,int64
rating_datetime,datetime64[ns]


,missing_rate
userId,0.0
movieId,0.0
rating,0.0
timestamp,0.0
rating_datetime,0.0


... (5 outputs left out)

,missing_rate
top_cast,0.009910
director,0.002643
tmdbId,0.000000



keywords


,dtype
tmdbId,int64
keywords_list,object


,missing_rate
keywords_list,0.083792
tmdbId,0.000000


## Milestone 2 Starter: Integration, Transformation, and Preliminary ABT

Build a preliminary ABT. State exactly what one row means, what keys define one row, and what tables were joined or aggregated.

Critical integration reminder:

Before creating a movie-level ABT, aggregate ratings. Examples include:

- average rating by `movieId`
- rating count by `movieId`
- rating standard deviation by `movieId`
- first and last rating date by `movieId`
- distinct user count by `movieId`

After merging, validate that `movieId` is not duplicated in the ABT.

### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 2 written workspace

Use this cell to document the ABT design required by the Canvas Milestone 2 page.

- Final or preliminary unit of analysis:
- Primary key for one row:
- Tables aggregated before joining:
- Tables joined directly:
- Row-count validation after each major merge:
- New variables created:
- Remaining data quality concerns:



In [11]:
# COPY - Path A, cell 1 of 3: connect to your own Neon database
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# Find the top folder of the repository (the folder that contains README.md).
repo_root = Path.cwd()
if not (repo_root / "README.md").exists() and (repo_root.parent / "README.md").exists():
    repo_root = repo_root.parent

# Add an empty PROJECT_DATABASE_URL line to .env the first time. Git ignores .env.
env_path = repo_root / ".env"
env_text = env_path.read_text() if env_path.exists() else ""
if "PROJECT_DATABASE_URL" not in env_text:
    if env_text and not env_text.endswith("\n"):
        env_text += "\n"
    env_text += ("# Your own Neon database (Path A). Paste your connection string after the = sign.\n"
                 "PROJECT_DATABASE_URL=\n")
    env_path.write_text(env_text)
    print("Added an empty PROJECT_DATABASE_URL line to .env")

load_dotenv(env_path, override=True)
project_url = os.getenv("PROJECT_DATABASE_URL")

# Saved query results go here, so the notebook also runs without your .env (see Section 3.4).
SQL_RESULTS_DIR = repo_root / "outputs" / "sql_results"
SQL_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

if project_url:
    project_engine = create_engine(project_url)
    with project_engine.connect() as conn:
        conn.execute(text("SELECT 1;"))
    print("Connected to your own database. For safety, we do not print the connection string.")
else:
    project_engine = None
    print("PROJECT_DATABASE_URL is empty, so the database steps will be skipped.")
    print("Path A: open .env, paste your own Neon connection string after PROJECT_DATABASE_URL=, save, and run this cell again.")



Connected to your own database. For safety, we do not print the connection string.


In [12]:
# COPY - Path A, cell 2 of 3: upload a CSV file or a DataFrame to your database
def upload_table(source, table_name):
    """Upload a CSV file path or a DataFrame to your own database and check the row count."""
    if project_engine is None:
        print(f"Skipped uploading {table_name}: no PROJECT_DATABASE_URL in .env.")
        return
    df = source if isinstance(source, pd.DataFrame) else pd.read_csv(source, low_memory=False)
    df.to_sql(table_name, project_engine, if_exists="replace", index=False, chunksize=1000, method="multi")
    with project_engine.connect() as conn:
        db_rows = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
    status = "OK" if db_rows == len(df) else "CHECK: the row counts differ"
    print(f"{table_name}: {len(df):,} rows in Python, {db_rows:,} rows in the database. {status}")


# Fill in your own files. One line per table: upload_table(path_or_dataframe, "table_name")
# In your project, use your data files, for example:
#   upload_table(repo_root / "data" / "ratings.csv", "ratings")
#   upload_table(abt, "movie_abt")      # a DataFrame you built in Pandas
upload_table(repo_root / "data" / "movies_metadata.csv", "movies")
upload_table(repo_root / "data" / "ratings.csv", "ratings")

movies: 9,082 rows in Python, 9,082 rows in the database. OK
ratings: 99,810 rows in Python, 99,810 rows in the database. OK


In [13]:
# COPY - Path A, cell 3 of 3: run SQL on your database and save the result
def run_project_sql(query, result_name):
    """Run SQL on your own database and save the result. Without a connection, show the saved result."""
    result_path = SQL_RESULTS_DIR / f"{result_name}.csv"
    if project_engine is not None:
        result = pd.read_sql(text(query), project_engine)
        result.to_csv(result_path, index=False)
        print(f"Ran on your database. Saved the result to {result_path.relative_to(repo_root)}")
        return result
    if result_path.exists():
        print(f"No database connection here, so this is the saved result from {result_path.relative_to(repo_root)}")
        return pd.read_csv(result_path)
    print(f"Skipped {result_name}: no PROJECT_DATABASE_URL and no saved result yet.")
    return pd.DataFrame()


In [14]:
def run_query(query, engine):
    """Run a SQL query when it has been filled in; keep blank practice cells runnable."""
    if not query or not query.strip():
        print("This practice cell is waiting for your SQL. Add your query inside the triple quotes, then run it again.")
        return pd.DataFrame()
    return pd.read_sql(query, engine)

Below I am merging Movies and Ratings datasets only keeping needed columns. I used movieId to match the two datasets and chose an inner join because I only wanted to keep movies that had audience ratings. This allows me to focus on movies with rating information for my analysis. 

In [15]:

query = '''
WITH movies_ratings AS (
SELECT
    m."movieId",
    m.title,
    m."primary_genre",
    r.rating,
    m.budget,
    m.revenue
    FROM movies AS m
INNER JOIN ratings AS r
    ON m."movieId" = r."movieId"
)
SELECT
    mr.*
FROM movies_ratings AS mr;
'''

movie_rating_merged = run_query(query, project_engine)

display(movie_rating_merged)

,movieId,title,primary_genre,rating,budget,revenue
0,31,Dangerous Minds,Drama,2.5,0,180000000.0
1,1029,Dumbo,Animation,3.0,812000,1600000.0
2,1061,Sleepers,Crime,3.0,44000000,165615285.0
3,1129,Escape from New York,Science Fiction,2.0,6000000,50244700.0
4,1172,Cinema Paradiso,Drama,4.0,0,11990401.0
...,...,...,...,...,...,...
99805,6268,Raising Victor Vargas,Drama,2.5,800000,2816116.0
99806,6269,Stevie,Documentary,4.0,0,97000.0
99807,6365,The Matrix Reloaded,Adventure,4.0,150000000,738599701.0
99808,6385,Whale Rider,Drama,2.5,8000000,41400000.0


After merging the movies and ratings datasets, I removed rows with missing values using dropna(). I chose to keep movies with zero budget or revenue because I want to compare which primary genres preform better based on audience ratings and profit. However, I understand that some zero values may represent missing financial information, which could affect my profit calculations.

In [18]:
#clean the movie_rating_merged DataFrame and drop missing values
movie_rating_merged_clean = movie_rating_merged.dropna()
movie_rating_merged_clean = movie_rating_merged_clean[(movie_rating_merged_clean['budget'] > 0) & (movie_rating_merged_clean['revenue'] > 0)]
display(movie_rating_merged_clean)

,movieId,title,primary_genre,rating,budget,revenue
1,1029,Dumbo,Animation,3.0,812000,1600000.0
2,1061,Sleepers,Crime,3.0,44000000,165615285.0
3,1129,Escape from New York,Science Fiction,2.0,6000000,50244700.0
5,1263,The Deer Hunter,Drama,2.0,15000000,50000000.0
6,1287,Ben-Hur,Action,2.0,15000000,146900000.0
...,...,...,...,...,...,...
99804,6212,Bringing Down the House,Comedy,2.5,20000000,132675402.0
99805,6268,Raising Victor Vargas,Drama,2.5,800000,2816116.0
99807,6365,The Matrix Reloaded,Adventure,4.0,150000000,738599701.0
99808,6385,Whale Rider,Drama,2.5,8000000,41400000.0


I grouped the movieId, title, and primary genre to calculate the average audience rating for each mvie. Since some movies had multiple ratings, grouping the data allowed me to have one row per movie instead of multiple rows. I then created a profit column by subtracting budget from revenue. This resulted in an ABT with 3,829 rows and 6 columns. 

In [39]:
#Taking movie_rating_merged_clean DataFrame and calculating the average rating per movie, and calculating profit (revenue - budget) in an abt
movie_rating_merged_abt = movie_rating_merged_clean.groupby(
    ["movieId", "title", "primary_genre"], as_index=False
).agg(
    rating_avg=("rating", "mean"),
    budget=("budget", "first"),
    revenue=("revenue", "first"),
)
movie_rating_merged_abt["profit_proxy"] = (
    movie_rating_merged_abt["revenue"] - movie_rating_merged_abt["budget"]
)

movie_rating_merged_abt = movie_rating_merged_abt.reset_index(drop=True)
display(movie_rating_merged_abt)

,movieId,title,primary_genre,rating_avg,budget,revenue,profit_proxy
0,1,Toy Story,Animation,3.872470,30000000,373554033.0,343554033.0
1,2,Jumanji,Adventure,3.401869,65000000,262797249.0,197797249.0
2,4,Waiting to Exhale,Comedy,2.384615,16000000,81452156.0,65452156.0
3,6,Heat,Action,3.884615,60000000,187436818.0,127436818.0
4,9,Sudden Death,Action,3.150000,35000000,64350171.0,29350171.0
...,...,...,...,...,...,...,...
3824,160438,Jason Bourne,Action,4.250000,120000000,415484914.0,295484914.0
3825,160563,The Legend of Tarzan,Action,2.500000,180000000,356743061.0,176743061.0
3826,160565,The Purge: Election Year,Action,2.000000,10000000,118587880.0,108587880.0
3827,160567,Mike and Dave Need Wedding Dates,Comedy,4.000000,33000000,77068246.0,44068246.0


In [21]:
print("Movies Before Merge", len(movies))
print("Ratings Before Merge", len(ratings))
print("Movies_Ratings after Merge", len(movie_rating_merged))
print("Movies_Ratings after Clean", len(movie_rating_merged_clean))
print("Unique movie IDs in movie_rating_merged_abt:", movie_rating_merged_abt["movieId"].nunique())
print("Duplicate movie IDs in movie_rating_merged_abt:", movie_rating_merged_abt["movieId"].duplicated().sum())
print("Missing values in movie_rating_merged_abt:", movie_rating_merged_abt.isnull().sum().sum())

Movies Before Merge 9082
Ratings Before Merge 99810
Movies_Ratings after Merge 99810
Movies_Ratings after Clean 76873
Unique movie IDs in movie_rating_merged_abt: 3829
Duplicate movie IDs in movie_rating_merged_abt: 0
Missing values in movie_rating_merged_abt: 0


Provided

### SQL Validation Connected to the ABT

The project milestone asks for database or SQL work that supports the workflow. This in-memory SQLite check verifies the movie-level ABT row count and key uniqueness without creating another permanent database file.


In [26]:
import sqlite3

with sqlite3.connect(":memory:") as connection:
    abt.to_sql("movies_abt", connection, if_exists="replace", index=False)
    sql_abt_check = pd.read_sql_query(
        """
        SELECT
            COUNT(*) AS abt_rows,
            COUNT(DISTINCT movieId) AS unique_movie_ids,
            SUM(CASE WHEN high_user_rating IS NULL THEN 1 ELSE 0 END) AS target_not_defined_rows
        FROM movies_abt;
        """,
        connection,
    )

sql_abt_check


,abt_rows,unique_movie_ids,target_not_defined_rows
0,9082,9082,7782


### Data Dictionary Starter

Complete the text fields for every column you keep in the final ABT. Remove unused columns before finalizing the dictionary so the documentation and exported ABT stay aligned.


In [40]:
data_dictionary = pd.DataFrame({
    "column_name": movie_rating_merged_abt.columns,
    "data_type": [str(movie_rating_merged_abt[column].dtype) for column in movie_rating_merged_abt.columns],
    "description": "",
    "source": "",
    "transformation_or_derivation": "",
    "notes_or_concerns": "",
})

data_dictionary.loc[data_dictionary["column_name"]== "movieId",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Unique identifier for each movie",
    "movies_metadata.csv",
    "Derived from the 'movieId' column in the movies_metadata.csv file",
    "Ensure that the 'movieId' values are unique and consistent across all tables."]
data_dictionary.loc[data_dictionary["column_name"]== "title",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Title of the movie",
    "movies_metadata.csv",
    "Derived from the 'title' column in the movies_metadata.csv file",
    "Ensure that the 'title' values are accurate and match the corresponding 'movieId'."]
data_dictionary.loc[data_dictionary["column_name"]== "primary_genre",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Primary genre of the movie",
    "movies_metadata.csv",
    "Derived from the 'primary_genre' column in the movies_metadata.csv file",
    "Ensure that the 'primary_genre' values are consistent and correctly represent the movie's genre."]
data_dictionary.loc[data_dictionary["column_name"]== "rating_avg",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Average user rating for the movie",
    "ratings.csv",
    "Calculated as the mean of the 'rating' values for each movie in the ratings.csv file",
    "Ensure that the 'rating_avg' values are accurate and reflect the overall user sentiment."]
data_dictionary.loc[data_dictionary["column_name"]== "budget",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Budget of the movie",
    "movies_metadata.csv",
    "Derived from the 'budget' column in the movies_metadata.csv file",
    "Ensure that the 'budget' values are accurate and represent the actual production costs."]
data_dictionary.loc[data_dictionary["column_name"]== "revenue",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Revenue generated by the movie",
    "movies_metadata.csv",
    "Derived from the 'revenue' column in the movies_metadata.csv file",
    "Ensure that the 'revenue' values are accurate and represent the actual earnings from the movie."]
data_dictionary.loc[data_dictionary["column_name"]== "profit_proxy",
        ["description", "source", "transformation_or_derivation", "notes_or_concerns"]] = [
    "Profit generated by the movie",
    "Calculated as revenue - budget",
    "Derived from the 'revenue' and 'budget' columns in the movies_metadata.csv file",
    "Ensure that the 'profit' values are accurate and reflect the actual financial performance of the movie."]

# Hint: use .loc[] to document one column at a time, following the Week 11 pattern.
data_dictionary


,column_name,data_type,description,source,transformation_or_derivation,notes_or_concerns
0,movieId,int64,Unique identifier for each movie,movies_metadata.csv,Derived from the 'movieId' column in the movie...,Ensure that the 'movieId' values are unique an...
1,title,object,Title of the movie,movies_metadata.csv,Derived from the 'title' column in the movies_...,Ensure that the 'title' values are accurate an...
2,primary_genre,object,Primary genre of the movie,movies_metadata.csv,Derived from the 'primary_genre' column in the...,Ensure that the 'primary_genre' values are con...
3,rating_avg,float64,Average user rating for the movie,ratings.csv,Calculated as the mean of the 'rating' values ...,Ensure that the 'rating_avg' values are accura...
4,budget,int64,Budget of the movie,movies_metadata.csv,Derived from the 'budget' column in the movies...,Ensure that the 'budget' values are accurate a...
5,revenue,float64,Revenue generated by the movie,movies_metadata.csv,Derived from the 'revenue' column in the movie...,Ensure that the 'revenue' values are accurate ...
6,profit_proxy,float64,Profit generated by the movie,Calculated as revenue - budget,Derived from the 'revenue' and 'budget' column...,Ensure that the 'profit' values are accurate a...


## Milestone 3 Starter: Prepared Dataset and Initial Model Application

Choose a target/outcome and features. Start simple. Your first model should be correct and interpretable before it becomes complex.

Modeling reminder:

- If you model `high_user_rating`, use only rows where that nullable target is defined; these movies have at least 20 ratings.
- State when the prediction is meant to occur. For a pre-release question, exclude post-release fields such as revenue, popularity, vote count, rating summaries, and rating dates.
- Do not interpret a rating model as proving artistic quality or causal effects.


### Read the Canvas milestone page first

Before you start this section, read the matching Canvas milestone assignment page. The Canvas page is the authoritative checklist for what you need to submit, how the milestone is graded, and what evidence should appear in your notebook/report. Use this starter section as a workspace, not as a replacement for the Canvas instructions.



### Milestone 3 written workspace

Use this cell to document the prepared dataset and initial model/analysis required by the Canvas Milestone 3 page.

- Target/outcome:
- Feature variables:
- Rows included/excluded and why:
- Train/test split or evaluation plan:
- Baseline model or analysis:
- Initial metrics/results:
- Interpretation limits:



In [30]:
# Milestone 3 code workspace
# Suggested sequence:
# 1. State the prediction or analysis timing in markdown.
# 2. Filter to rows where your target is defined.
# 3. Select only features available at that time.
# 4. Check class balance or the target distribution.
# 5. Create a reproducible train/test split when modeling.
# 6. Fit a simple baseline and evaluate it on test data.


1. State the prediction or analysis timing in markdown
    - The prupose of this is to determine whether primary movie genre can predict audience ratings and financial performance before a movie is released. Primary genre was selected as the main feature because it is information that would be available before release. Average audience ratings and the revenue-budget proxy will be used as 2 seperate target variables to determine whether certain genres tend to perform better in one outcome then the other.

The 2 targets selected for this analysis are average audience ratings (rating_avg) and financial performance (profit_proxy). The purpose of selecting two targets is to determine whether the primary genres that recieve higher audience ratings also tend to perform better financially. Primary genre was seleced as the only feature because it directly relates to the reasearch question and is information that would be avaiable before a movie is released.

In [42]:
#defind the target variable and features for the model

target_rating = "rating_avg"
target_profit = "profit_proxy"

features = [
    "primary_genre"]

In [50]:
# Dataframe for predicting movie ratings
model_rating_df = movie_rating_merged_abt.dropna(subset=[target_rating]+features)

#dataframe for predicting movie profit
model_profit_df = movie_rating_merged_abt.dropna(subset=[target_profit]+features)

print("Number of rows in model_rating_df:", len(model_rating_df))
print("Number of rows in model_profit_df:", len(model_profit_df))

Number of rows in model_rating_df: 3829
Number of rows in model_profit_df: 3829


In [51]:
display(model_rating_df.describe(include='all'))
display(model_profit_df.describe(include='all'))

,movieId,title,primary_genre,rating_avg,budget,revenue,profit_proxy
count,3829.000000,3829,3829,3829.000000,3.829000e+03,3.829000e+03,3.829000e+03
unique,NaN,3764,18,NaN,NaN,NaN,NaN
top,NaN,King Kong,Drama,NaN,NaN,NaN,NaN
freq,NaN,3,859,NaN,NaN,NaN,NaN
mean,34865.780099,NaN,NaN,3.208032,3.682516e+07,1.123083e+08,7.548309e+07
std,42098.839145,NaN,NaN,0.769231,4.306957e+07,1.818301e+08,1.539617e+08
min,1.000000,NaN,NaN,0.500000,1.000000e+00,1.000000e+00,-1.657101e+08
25%,2648.000000,NaN,NaN,2.750000,8.000000e+06,1.436340e+07,4.767930e+05
50%,6537.000000,NaN,NaN,3.318182,2.200000e+07,4.661607e+07,2.194424e+07
75%,67252.000000,NaN,NaN,3.750000,5.000000e+07,1.303589e+08,8.660323e+07


,movieId,title,primary_genre,rating_avg,budget,revenue,profit_proxy
count,3829.000000,3829,3829,3829.000000,3.829000e+03,3.829000e+03,3.829000e+03
unique,NaN,3764,18,NaN,NaN,NaN,NaN
top,NaN,King Kong,Drama,NaN,NaN,NaN,NaN
freq,NaN,3,859,NaN,NaN,NaN,NaN
mean,34865.780099,NaN,NaN,3.208032,3.682516e+07,1.123083e+08,7.548309e+07
std,42098.839145,NaN,NaN,0.769231,4.306957e+07,1.818301e+08,1.539617e+08
min,1.000000,NaN,NaN,0.500000,1.000000e+00,1.000000e+00,-1.657101e+08
25%,2648.000000,NaN,NaN,2.750000,8.000000e+06,1.436340e+07,4.767930e+05
50%,6537.000000,NaN,NaN,3.318182,2.200000e+07,4.661607e+07,2.194424e+07
75%,67252.000000,NaN,NaN,3.750000,5.000000e+07,1.303589e+08,8.660323e+07


Target Distribution
- Both modeling Dataframes contain 3,829 movies across 18 primary genres, allowing for a consistent comparison between the two outcomes.
- The average audience rating is 3.21 out of 5, with ratings ranging from 0.5 to 5.
- Financial performance has a mean of approximately $75.48 million and a median of $21.94 million, suggesting that movie with extremely high financial returns may influence the average.

- Data Leakage Considerations
    - To prevent data leakage, revnue will be excluded as a feature because it is not known before a movie is released and is directly used to calculate profit_proxy. Neither target variable will be used to predict the other because outcomes are determined after release. Budget, movieId, and title will also be excluded because the analysis focuses specifically on primary genre.

- Method Selection
    - Linear regression was selected because both target variables, rating_avg and profit_proxy, are continuous numerical values. Two seperate regression models will allow me to compare predicted audience ratings and financial performance across primary genres. A baseline mdoel will also be used to determine whether including primary genre improves prediction accuracy.

In [52]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [53]:
#define variables for the rating model
X_rating = model_rating_df[features]
y_rating = model_rating_df[target_rating]

#defibd variables for the profit model
X_profit = model_profit_df[features]
y_profit = model_profit_df[target_profit]


In [55]:
display(X_rating.head())
display(y_rating.head())

display(X_profit.head())
display(y_profit.head())

,primary_genre
0,Animation
1,Adventure
2,Comedy
3,Action
4,Action


0    3.872470
1    3.401869
2    2.384615
3    3.884615
4    3.150000
Name: rating_avg, dtype: float64

,primary_genre
0,Animation
1,Adventure
2,Comedy
3,Action
4,Action


0    343554033.0
1    197797249.0
2     65452156.0
3    127436818.0
4     29350171.0
Name: profit_proxy, dtype: float64

- Two seperate sets of x and y variables were created for the regression models. Primary genre is the only feature for both models, while avg_rating and profit_proxy are the respective target variables. The results confirm that the correct feature and targets were selected. Since primary genre is a caterogical variable, it will need to be converted into a numerical format before running the linear regression models.

In [58]:
# train/test split for the rating model
X_rating_train, X_rating_test, y_rating_train, y_rating_test = train_test_split(
    X_rating, y_rating, test_size=0.2, random_state=42)

# train/test split for the profit model
X_profit_train, X_profit_test, y_profit_train, y_profit_test = train_test_split(
    X_profit, y_profit, test_size=0.2, random_state=42)

print("Rating training rows:", len(X_rating_train))
print("Rating test rows:", len(X_rating_test))
print("Profit training rows:", len(X_profit_train))
print("Profit test rows:", len(X_profit_test))

Rating training rows: 3063
Rating test rows: 766
Profit training rows: 3063
Profit test rows: 766


## Final Project Starter: Interpretation, Ethics, and Recommendations

Use this section to connect your analysis back to a business or research decision.

Address:

- Main finding
- What the model/analysis can and cannot support
- Limitations
- Ethics or responsible use concerns
- 2-3 cautious recommendations

### Read the Canvas final submission page first

Before completing this section, read the Canvas final submission assignment page. The Canvas page is the authoritative checklist for the final notebook, HTML report, required files, and submission format. Use this section to organize your final evidence and interpretation.



### Final report written workspace

Use this cell to draft the final interpretation required by the Canvas final submission page.

- Main finding:
- Evidence supporting the finding:
- Business or research implication:
- Limitations:
- Ethics/responsible use concerns:
- Recommendations:
- What you would improve with more time:



In [32]:
# Final project code workspace
# Save final datasets, figures, or tables here when they are ready.
# Recommended filenames:
# final_abt_path = OUTPUT_DIR / "movies_final_abt.csv"
# dictionary_path = OUTPUT_DIR / "movies_data_dictionary.csv"
# metrics_path = OUTPUT_DIR / "movies_model_metrics.csv"
# report_path = REPORTS_DIR / "movies_final_report.html"


In [33]:
# Save final deliverables only after your ABT and documentation are complete.
# abt.to_csv(OUTPUT_DIR / "movies_final_abt.csv", index=False)
# data_dictionary.to_csv(OUTPUT_DIR / "movies_data_dictionary.csv", index=False)

# Before final submission, restart the kernel, run all cells, and export the notebook to HTML.
